In [9]:
from pathlib import Path
import numpy as np
import pandas as pd


OUTPUT_DIR = Path("/Users/zsh/Downloads/PROJECT/code")

Q5_FILES = [
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2017~2018/2017_data_179_activities.csv",
        "year": 3,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Siyan Xin/2018~2019/2018_data_179_activities.csv",
        "year": 4,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/1920_london32_stable179.csv",
        "year": 5,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/code/2021_london32_stable179.csv",
        "year": 6,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year7_179activities.csv",
        "year": 7,
    },
    {
        "path": "/Users/zsh/Downloads/PROJECT/London-Sport2-Group20/Jingyi Hua/data/processed/year8_179activities.csv",
        "year": 8,
    },
]

REQUIRED_COLUMNS = [
    "LA_2023",
    "Age16plus",
    "Age9",
    "VolFrqB_Pop",
    "MEMS7GR_ALL",
    "wt_final_online",
]

MISSING_CODES = [
    -99,
    -98,
    -97,
    -96,
    -95,
    -94,
    -93,
    -92,
    -91,
]

WEIGHT_COL = "wt_final_online"

# Define thresholds used to flag unreliable cells.
MIN_CELL_N = 30
MIN_FREQUENT_N = 5

AGE9_LABELS = {
    2: "16-24",
    3: "25-34",
    4: "35-44",
    5: "45-54",
    6: "55-64",
    7: "65-74",
    8: "75-84",
    9: "85+",
}

VOL_BINARY_A_LABELS = {
    0: "not_twiceplus_volunteer",
    1: "twiceplus_volunteer",
}

ACTIVITY_LEVEL_LABELS = {
    0: "inactive",
    1: "fairly_active",
    2: "active",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
def load_q5_master(file_info_list):
    yearly_dataframes = []

    for file_info in file_info_list:
        file_path = Path(file_info["path"])
        year_value = file_info["year"]

        if not file_path.exists():
            raise FileNotFoundError(
                f"File not found: {file_path}"
            )

        df = pd.read_csv(
            file_path,
            low_memory=False,
        )

        missing_columns = [
            column
            for column in REQUIRED_COLUMNS
            if column not in df.columns
        ]

        if missing_columns:
            raise ValueError(
                f"Missing required columns in "
                f"{file_path.name}: {missing_columns}"
            )

        df = df[REQUIRED_COLUMNS].copy()

        df["year"] = year_value

        yearly_dataframes.append(df)

    master_df = pd.concat(
        yearly_dataframes,
        ignore_index=True,
    )

    master_df = master_df.replace(
        MISSING_CODES,
        np.nan,
    )

    numeric_columns = [
        "LA_2023",
        "Age16plus",
        "Age9",
        "VolFrqB_Pop",
        "MEMS7GR_ALL",
        WEIGHT_COL,
    ]

    for column in numeric_columns:
        master_df[column] = pd.to_numeric(
            master_df[column],
            errors="coerce",
        )

    master_df = master_df.loc[
        master_df["Age16plus"] == 1
    ].copy()

    master_df["LA_2023"] = master_df[
        "LA_2023"
    ].astype("Int64")

    master_df["year"] = master_df[
        "year"
    ].astype("Int64")

    print(
        f"Done: {len(master_df):,} respondent rows, "
        f"{master_df['year'].nunique()} years, "
        f"{master_df['LA_2023'].nunique()} boroughs"
    )

    return master_df

q5_master = load_q5_master(
    file_info_list=Q5_FILES
)

print(q5_master.shape)

print("\nRows by year:")
print(
    q5_master["year"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nNumber of boroughs:")
print(q5_master["LA_2023"].nunique())

print("\nOriginal Age9 values:")
print(
    q5_master["Age9"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nOriginal MEMS7GR_ALL values:")
print(
    q5_master["MEMS7GR_ALL"]
    .value_counts(dropna=False)
    .sort_index()
)

Done: 96,629 respondent rows, 6 years, 32 boroughs
(96629, 7)

Rows by year:
year
3    15967
4    15889
5    16091
6    16028
7    16139
8    16515
Name: count, dtype: Int64

Number of boroughs:
32

Original Age9 values:
Age9
2.0     7883
3.0    18022
4.0    19933
5.0    16445
6.0    14753
7.0    11772
8.0     5556
9.0     1386
NaN      879
Name: count, dtype: int64

Original MEMS7GR_ALL values:
MEMS7GR_ALL
0.0    21824
1.0    10351
2.0    64454
Name: count, dtype: int64


In [11]:
def add_q5_analysis_variables(df):
    result = df.copy()

    result["age_group_code"] = pd.to_numeric(
        result["Age9"],
        errors="coerce",
    ).astype("Int64")

    result["age_group"] = result[
        "age_group_code"
    ].map(AGE9_LABELS)

    early_years = result["year"].isin([3, 4])
    later_years = result["year"].isin([5, 6, 7, 8])

    # Create the six-year comparable volunteering variable.
    result["vol_binary_A"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )

    result.loc[
        early_years
        & (result["VolFrqB_Pop"] == 0),
        "vol_binary_A",
    ] = 0

    result.loc[
        early_years
        & (result["VolFrqB_Pop"] == 1),
        "vol_binary_A",
    ] = 1

    result.loc[
        later_years
        & result["VolFrqB_Pop"].isin([0, 1]),
        "vol_binary_A",
    ] = 0

    result.loc[
        later_years
        & result["VolFrqB_Pop"].isin([2, 3, 4]),
        "vol_binary_A",
    ] = 1

    result["vol_binary_A_label"] = result[
        "vol_binary_A"
    ].map(VOL_BINARY_A_LABELS)

    result["activity_level_3cat"] = pd.Series(
        pd.NA,
        index=result.index,
        dtype="Int64",
    )

    valid_activity = result[
        "MEMS7GR_ALL"
    ].isin([0, 1, 2])

    result.loc[
        valid_activity,
        "activity_level_3cat",
    ] = (
        result.loc[
            valid_activity,
            "MEMS7GR_ALL",
        ]
        .astype(int)
    )

    result[
        "activity_level_3cat_label"
    ] = result["activity_level_3cat"].map(
        ACTIVITY_LEVEL_LABELS
    )


    return result

q5_master = add_q5_analysis_variables(
    q5_master
)

print("\nVersion A volunteering variable:")
print(
    q5_master["vol_binary_A"]
    .value_counts(dropna=False)
    .sort_index()
)


Version A volunteering variable:
vol_binary_A
0       64910
1        7529
<NA>    24190
Name: count, dtype: Int64


In [12]:
def prepare_q5_analysis_sample(
    df,
    weight_col,
):
    result = df.copy()

    eligible = (
        result["year"].isin([3, 4, 5, 6, 7, 8])
        & result["LA_2023"].notna()
        & result["age_group_code"].isin(
            AGE9_LABELS.keys()
        )
        & result["activity_level_3cat"].isin(
            [0, 1, 2]
        )
        & result["vol_binary_A"].isin([0, 1])
        & result[weight_col].notna()
        & (result[weight_col] > 0)
    )

    analysis_sample = result.loc[
        eligible
    ].copy()

    # Create helper variables for weighted aggregation.
    analysis_sample["weighted_total"] = (
        analysis_sample[weight_col]
    )

    analysis_sample["frequent_case"] = (
        analysis_sample["vol_binary_A"] == 1
    ).astype(int)

    analysis_sample["not_frequent_case"] = (
        analysis_sample["vol_binary_A"] == 0
    ).astype(int)

    analysis_sample["weighted_frequent"] = (
        analysis_sample[weight_col]
        * analysis_sample["frequent_case"]
    )

    analysis_sample["weighted_not_frequent"] = (
        analysis_sample[weight_col]
        * analysis_sample["not_frequent_case"]
    )

    analysis_sample["weight_squared"] = (
        analysis_sample[weight_col] ** 2
    )

    print(
        f"Eligible respondent rows: "
        f"{len(analysis_sample):,}"
    )

    print(
        f"Excluded respondent rows: "
        f"{len(result) - len(analysis_sample):,}"
    )

    return analysis_sample

q5_analysis_sample = prepare_q5_analysis_sample(
    df=q5_master,
    weight_col=WEIGHT_COL,
)

print(q5_analysis_sample.shape)

print("\nEligible rows by year:")
print(
    q5_analysis_sample["year"]
    .value_counts()
    .sort_index()
)

print("\nEligible rows by activity level:")
print(
    q5_analysis_sample[
        "activity_level_3cat_label"
    ].value_counts()
)

Eligible respondent rows: 58,850
Excluded respondent rows: 37,779
(58850, 19)

Eligible rows by year:
year
3     5082
4     5455
5    11677
6    11930
7    12081
8    12625
Name: count, dtype: Int64

Eligible rows by activity level:
activity_level_3cat_label
active           39926
inactive         12679
fairly_active     6245
Name: count, dtype: int64


In [13]:
def make_q5_observed_panel(
    analysis_sample,
    weight_col,
    min_cell_n=30,
    min_frequent_n=5,
):
    group_columns = [
        "year",
        "LA_2023",
        "age_group_code",
        "activity_level_3cat",
    ]

    observed_panel = (
        analysis_sample
        .groupby(
            group_columns,
            dropna=False,
            observed=True,
        )
        .agg(
            n_parent_cell=(
                "vol_binary_A",
                "size",
            ),
            weighted_n_parent_cell=(
                weight_col,
                "sum",
            ),
            sum_weight_squared=(
                "weight_squared",
                "sum",
            ),
            n_frequent_volunteer=(
                "frequent_case",
                "sum",
            ),
            weighted_n_frequent_volunteer=(
                "weighted_frequent",
                "sum",
            ),
            n_not_frequent_volunteer=(
                "not_frequent_case",
                "sum",
            ),
            weighted_n_not_frequent_volunteer=(
                "weighted_not_frequent",
                "sum",
            ),
        )
        .reset_index()
    )

    # Calculate the weighted twice-plus volunteering rate.
    observed_panel[
        "frequent_volunteer_rate"
    ] = (
        observed_panel[
            "weighted_n_frequent_volunteer"
        ]
        / observed_panel[
            "weighted_n_parent_cell"
        ]
    )

    # Calculate the complementary not-twice-plus rate.
    observed_panel[
        "not_frequent_volunteer_rate"
    ] = (
        observed_panel[
            "weighted_n_not_frequent_volunteer"
        ]
        / observed_panel[
            "weighted_n_parent_cell"
        ]
    )

    # Calculate the effective sample size from the survey weights.
    observed_panel[
        "effective_n_parent_cell"
    ] = (
        observed_panel[
            "weighted_n_parent_cell"
        ] ** 2
        / observed_panel[
            "sum_weight_squared"
        ]
    )

    observed_panel["age_group"] = observed_panel[
        "age_group_code"
    ].map(AGE9_LABELS)

    observed_panel[
        "activity_level_3cat_label"
    ] = observed_panel[
        "activity_level_3cat"
    ].map(ACTIVITY_LEVEL_LABELS)

    # Flag cells with limited raw sample sizes.
    observed_panel["small_cell"] = (
        observed_panel["n_parent_cell"]
        < min_cell_n
    )

    # Flag cells with fewer than the required number of positive cases.
    observed_panel["few_frequent_cases"] = (
        observed_panel[
            "n_frequent_volunteer"
        ]
        < min_frequent_n
    )

    # Identify cells containing no twice-plus volunteers.
    observed_panel["zero_frequent_cases"] = (
        observed_panel[
            "n_frequent_volunteer"
        ]
        == 0
    )

    observed_panel["cell_observed"] = True
    observed_panel["rate_available"] = True

    observed_panel = observed_panel.sort_values(
        [
            "year",
            "LA_2023",
            "age_group_code",
            "activity_level_3cat",
        ]
    ).reset_index(drop=True)

    print(
        f"Done: {len(observed_panel):,} "
        f"observed panel rows"
    )

    return observed_panel

q5_observed_panel = make_q5_observed_panel(
    analysis_sample=q5_analysis_sample,
    weight_col=WEIGHT_COL,
    min_cell_n=MIN_CELL_N,
    min_frequent_n=MIN_FREQUENT_N,
)

q5_observed_panel.head(10)

print("\nObserved cells by year:")
print(
    q5_observed_panel["year"]
    .value_counts()
    .sort_index()
)

print("\nSmall-cell distribution:")
print(
    q5_observed_panel["small_cell"]
    .value_counts(dropna=False)
)

print("\nCells with few frequent volunteers:")
print(
    q5_observed_panel[
        "few_frequent_cases"
    ].value_counts(dropna=False)
)

Done: 4,127 observed panel rows

Observed cells by year:
year
3    647
4    652
5    695
6    704
7    713
8    716
Name: count, dtype: Int64

Small-cell distribution:
small_cell
True     3521
False     606
Name: count, dtype: Int64

Cells with few frequent volunteers:
few_frequent_cases
True     3669
False     458
Name: count, dtype: int64


In [14]:
q5_observed_panel[
    [
        "n_parent_cell",
        "weighted_n_parent_cell",
        "effective_n_parent_cell",
    ]
].describe()

,n_parent_cell,weighted_n_parent_cell,effective_n_parent_cell
count,4127.0,4127.000000,4127.000000
mean,14.259753,22.598999,9.555836
std,16.802693,26.160528,10.721698
min,1.0,0.143605,1.000000
25%,3.0,5.740179,2.696573
50%,8.0,13.066194,5.550681
75%,18.0,30.058807,11.987554
max,144.0,262.315341,95.823022


In [15]:
def complete_q5_panel(
    observed_panel,
    master_df,
    min_cell_n=30,
    min_frequent_n=5,
):
    years = [3, 4, 5, 6, 7, 8]

    boroughs = sorted(
        master_df["LA_2023"]
        .dropna()
        .astype(int)
        .unique()
    )

    age_group_codes = sorted(
        AGE9_LABELS.keys()
    )

    activity_level_codes = sorted(
        ACTIVITY_LEVEL_LABELS.keys()
    )

    # Create every theoretical year-borough-age-activity combination.
    complete_index = pd.MultiIndex.from_product(
        [
            years,
            boroughs,
            age_group_codes,
            activity_level_codes,
        ],
        names=[
            "year",
            "LA_2023",
            "age_group_code",
            "activity_level_3cat",
        ],
    )

    complete_grid = (
        complete_index
        .to_frame(index=False)
    )

    key_columns = [
        "year",
        "LA_2023",
        "age_group_code",
        "activity_level_3cat",
    ]

    measure_columns = [
        "frequent_volunteer_rate",
        "not_frequent_volunteer_rate",
        "n_parent_cell",
        "weighted_n_parent_cell",
        "sum_weight_squared",
        "effective_n_parent_cell",
        "n_frequent_volunteer",
        "weighted_n_frequent_volunteer",
        "n_not_frequent_volunteer",
        "weighted_n_not_frequent_volunteer",
        "cell_observed",
    ]

    complete_panel = complete_grid.merge(
        observed_panel[
            key_columns + measure_columns
        ],
        on=key_columns,
        how="left",
        validate="one_to_one",
    )

    # Add readable category labels.
    complete_panel["age_group"] = complete_panel[
        "age_group_code"
    ].map(AGE9_LABELS)

    complete_panel[
        "activity_level_3cat_label"
    ] = complete_panel[
        "activity_level_3cat"
    ].map(ACTIVITY_LEVEL_LABELS)

    # Convert the merged indicator before filling missing values.
    complete_panel["cell_observed"] = (
        complete_panel["cell_observed"]
        .astype("boolean")
        .fillna(False)
        .astype(bool)
    )

    raw_count_columns = [
        "n_parent_cell",
        "n_frequent_volunteer",
        "n_not_frequent_volunteer",
    ]

    weighted_count_columns = [
        "weighted_n_parent_cell",
        "sum_weight_squared",
        "weighted_n_frequent_volunteer",
        "weighted_n_not_frequent_volunteer",
    ]

    # An unobserved cell contains zero eligible respondents.
    for column in raw_count_columns:
        complete_panel[column] = (
            complete_panel[column]
            .fillna(0)
            .astype(int)
        )

    # Weighted counts are zero when no eligible respondent exists.
    for column in weighted_count_columns:
        complete_panel[column] = (
            complete_panel[column]
            .fillna(0.0)
        )

    # Effective sample size is undefined when no respondent exists.
    complete_panel.loc[
        ~complete_panel["cell_observed"],
        "effective_n_parent_cell",
    ] = np.nan

    # A rate is available only for an observed cell with a valid rate.
    complete_panel["rate_available"] = (
        complete_panel["cell_observed"]
        & complete_panel[
            "frequent_volunteer_rate"
        ].notna()
    )

    observed_mask = complete_panel[
        "cell_observed"
    ]

    # Reliability flags are only defined for observed cells.
    complete_panel["small_cell"] = pd.Series(
        pd.NA,
        index=complete_panel.index,
        dtype="boolean",
    )

    complete_panel.loc[
        observed_mask,
        "small_cell",
    ] = (
        complete_panel.loc[
            observed_mask,
            "n_parent_cell",
        ]
        < min_cell_n
    )

    complete_panel[
        "few_frequent_cases"
    ] = pd.Series(
        pd.NA,
        index=complete_panel.index,
        dtype="boolean",
    )

    complete_panel.loc[
        observed_mask,
        "few_frequent_cases",
    ] = (
        complete_panel.loc[
            observed_mask,
            "n_frequent_volunteer",
        ]
        < min_frequent_n
    )

    complete_panel[
        "zero_frequent_cases"
    ] = pd.Series(
        pd.NA,
        index=complete_panel.index,
        dtype="boolean",
    )

    complete_panel.loc[
        observed_mask,
        "zero_frequent_cases",
    ] = (
        complete_panel.loc[
            observed_mask,
            "n_frequent_volunteer",
        ]
        == 0
    )

    # Create one identifier for each borough-age-activity time series.
    complete_panel["series_id"] = (
        "borough_"
        + complete_panel["LA_2023"].astype(str)
        + "__age_"
        + complete_panel[
            "age_group_code"
        ].astype(str)
        + "__activity_"
        + complete_panel[
            "activity_level_3cat"
        ].astype(str)
    )

    complete_panel[
        "observed_years_in_series"
    ] = (
        complete_panel
        .groupby("series_id")[
            "cell_observed"
        ]
        .transform("sum")
        .astype(int)
    )

    complete_panel[
        "series_coverage_rate"
    ] = (
        complete_panel[
            "observed_years_in_series"
        ]
        / len(years)
    )

    # Identify series with at least one observed historical year.
    complete_panel[
        "series_has_history"
    ] = (
        complete_panel[
            "observed_years_in_series"
        ]
        > 0
    )

    # Identify series with all six historical years observed.
    complete_panel[
        "series_complete_history"
    ] = (
        complete_panel[
            "observed_years_in_series"
        ]
        == len(years)
    )

    final_columns = [
        "series_id",
        "year",
        "LA_2023",
        "age_group_code",
        "age_group",
        "activity_level_3cat",
        "activity_level_3cat_label",
        "frequent_volunteer_rate",
        "not_frequent_volunteer_rate",
        "n_parent_cell",
        "weighted_n_parent_cell",
        "sum_weight_squared",
        "effective_n_parent_cell",
        "n_frequent_volunteer",
        "weighted_n_frequent_volunteer",
        "n_not_frequent_volunteer",
        "weighted_n_not_frequent_volunteer",
        "small_cell",
        "few_frequent_cases",
        "zero_frequent_cases",
        "cell_observed",
        "rate_available",
        "observed_years_in_series",
        "series_coverage_rate",
        "series_has_history",
        "series_complete_history",
    ]

    complete_panel = (
        complete_panel[final_columns]
        .sort_values(
            [
                "LA_2023",
                "age_group_code",
                "activity_level_3cat",
                "year",
            ]
        )
        .reset_index(drop=True)
    )

    print(
        f"Done: {len(complete_panel):,} "
        f"rows in the complete panel"
    )

    print(
        f"Observed cells: "
        f"{complete_panel['cell_observed'].sum():,}"
    )

    print(
        f"Unobserved cells: "
        f"{(~complete_panel['cell_observed']).sum():,}"
    )

    return complete_panel


q5_complete_panel = complete_q5_panel(
    observed_panel=q5_observed_panel,
    master_df=q5_master,
    min_cell_n=MIN_CELL_N,
    min_frequent_n=MIN_FREQUENT_N,
)

q5_complete_panel.head(10)

Done: 4,608 rows in the complete panel
Observed cells: 4,127
Unobserved cells: 481


,series_id,year,LA_2023,age_group_code,age_group,activity_level_3cat,activity_level_3cat_label,frequent_volunteer_rate,not_frequent_volunteer_rate,n_parent_cell,...,weighted_n_not_frequent_volunteer,small_cell,few_frequent_cases,zero_frequent_cases,cell_observed,rate_available,observed_years_in_series,series_coverage_rate,series_has_history,series_complete_history
0,borough_8__age_2__activity_0,3,8,2,16-24,0,inactive,0.000000,1.000000,4,...,16.884929,True,True,True,True,True,6,1.0,True,True
1,borough_8__age_2__activity_0,4,8,2,16-24,0,inactive,0.000000,1.000000,3,...,3.676887,True,True,True,True,True,6,1.0,True,True
2,borough_8__age_2__activity_0,5,8,2,16-24,0,inactive,0.373935,0.626065,15,...,11.398282,True,True,False,True,True,6,1.0,True,True
3,borough_8__age_2__activity_0,6,8,2,16-24,0,inactive,0.040428,0.959572,14,...,13.160482,True,True,False,True,True,6,1.0,True,True
4,borough_8__age_2__activity_0,7,8,2,16-24,0,inactive,0.080576,0.919424,19,...,24.886254,True,True,False,True,True,6,1.0,True,True
5,borough_8__age_2__activity_0,8,8,2,16-24,0,inactive,0.000000,1.000000,20,...,31.771745,True,True,True,True,True,6,1.0,True,True
6,borough_8__age_2__activity_1,3,8,2,16-24,1,fairly_active,0.000000,1.000000,3,...,4.168186,True,True,True,True,True,6,1.0,True,True
7,borough_8__age_2__activity_1,4,8,2,16-24,1,fairly_active,0.000000,1.000000,5,...,4.242021,True,True,True,True,True,6,1.0,True,True
8,borough_8__age_2__activity_1,5,8,2,16-24,1,fairly_active,0.000000,1.000000,4,...,7.416873,True,True,True,True,True,6,1.0,True,True
9,borough_8__age_2__activity_1,6,8,2,16-24,1,fairly_active,0.000000,1.000000,10,...,12.955784,True,True,True,True,True,6,1.0,True,True


In [16]:
def validate_q5_panel(panel):
    key_columns = [
        "year",
        "LA_2023",
        "age_group_code",
        "activity_level_3cat",
    ]

    reliability_flag_columns = [
        "small_cell",
        "few_frequent_cases",
        "zero_frequent_cases",
    ]

    # Calculate the expected number of complete panel rows.
    expected_rows = (
        panel["year"].nunique()
        * panel["LA_2023"].nunique()
        * panel["age_group_code"].nunique()
        * panel["activity_level_3cat"].nunique()
    )

    # Check whether the panel key is unique.
    duplicate_count = panel.duplicated(
        subset=key_columns
    ).sum()

    if duplicate_count != 0:
        raise ValueError(
            f"Duplicate panel keys found: "
            f"{duplicate_count}"
        )

    # Check whether the complete panel contains all theoretical combinations.
    if len(panel) != expected_rows:
        raise ValueError(
            f"Expected {expected_rows} panel rows, "
            f"but found {len(panel)}"
        )

    # Check whether all 32 London boroughs are included.
    if panel["LA_2023"].nunique() != 32:
        raise ValueError(
            "The final dataset does not contain "
            "exactly 32 boroughs"
        )

    # Separate observed and unobserved cells.
    observed = panel.loc[
        panel["cell_observed"]
    ].copy()

    unobserved = panel.loc[
        ~panel["cell_observed"]
    ].copy()

    # Check that all observed cells contain both volunteering rates.
    if observed[
        "frequent_volunteer_rate"
    ].isna().any():
        raise ValueError(
            "Some observed cells have a missing "
            "frequent volunteer rate"
        )

    if observed[
        "not_frequent_volunteer_rate"
    ].isna().any():
        raise ValueError(
            "Some observed cells have a missing "
            "not-frequent volunteer rate"
        )

    # Check that all observed rates fall within the valid range.
    if not observed[
        "frequent_volunteer_rate"
    ].between(0, 1).all():
        raise ValueError(
            "Some frequent volunteer rates "
            "fall outside the 0-1 range"
        )

    if not observed[
        "not_frequent_volunteer_rate"
    ].between(0, 1).all():
        raise ValueError(
            "Some not-frequent volunteer rates "
            "fall outside the 0-1 range"
        )

    # Check that the two volunteering rates sum to one.
    rate_sum = (
        observed["frequent_volunteer_rate"]
        + observed["not_frequent_volunteer_rate"]
    )

    if not np.allclose(
        rate_sum,
        1.0,
        atol=1e-10,
    ):
        raise ValueError(
            "The two volunteering rates "
            "do not sum to one"
        )

    # Check that the two raw category counts equal the parent-cell count.
    raw_count_sum = (
        observed["n_frequent_volunteer"]
        + observed["n_not_frequent_volunteer"]
    )

    if not (
        raw_count_sum
        == observed["n_parent_cell"]
    ).all():
        raise ValueError(
            "Raw category counts do not equal "
            "the parent-cell count"
        )

    # Check that the two weighted category counts equal the weighted total.
    weighted_count_sum = (
        observed["weighted_n_frequent_volunteer"]
        + observed["weighted_n_not_frequent_volunteer"]
    )

    if not np.allclose(
        weighted_count_sum,
        observed["weighted_n_parent_cell"],
        atol=1e-10,
    ):
        raise ValueError(
            "Weighted category counts do not equal "
            "the weighted parent-cell count"
        )

    # Check that observed cells contain a valid effective sample size.
    if observed[
        "effective_n_parent_cell"
    ].isna().any():
        raise ValueError(
            "Some observed cells have a missing "
            "effective sample size"
        )

    if not (
        observed["effective_n_parent_cell"] > 0
    ).all():
        raise ValueError(
            "Observed effective sample sizes "
            "must be greater than zero"
        )

    # Effective sample size should not exceed the raw sample size.
    if not (
        observed["effective_n_parent_cell"]
        <= observed["n_parent_cell"] + 1e-10
    ).all():
        raise ValueError(
            "Effective sample size cannot exceed "
            "the raw cell sample size"
        )

    # Check that all observed cells contain reliability indicators.
    if observed[
        reliability_flag_columns
    ].isna().any().any():
        raise ValueError(
            "Observed cells must contain all "
            "reliability flags"
        )

    # Check that unobserved cells do not contain estimated rates.
    if unobserved[
        "frequent_volunteer_rate"
    ].notna().any():
        raise ValueError(
            "Unobserved cells must not contain "
            "a frequent volunteer rate"
        )

    if unobserved[
        "not_frequent_volunteer_rate"
    ].notna().any():
        raise ValueError(
            "Unobserved cells must not contain "
            "a not-frequent volunteer rate"
        )

    # Check that effective sample size remains missing for unobserved cells.
    if unobserved[
        "effective_n_parent_cell"
    ].notna().any():
        raise ValueError(
            "Unobserved cells must not contain "
            "an effective sample size"
        )

    # Check that reliability indicators remain missing for unobserved cells.
    if unobserved[
        reliability_flag_columns
    ].notna().any().any():
        raise ValueError(
            "Reliability flags must remain missing "
            "for unobserved cells"
        )

    # Check whether the rate availability indicator is consistent.
    expected_rate_available = (
        panel["cell_observed"]
        & panel["frequent_volunteer_rate"].notna()
    )

    if not (
        panel["rate_available"]
        == expected_rate_available
    ).all():
        raise ValueError(
            "The rate_available indicator "
            "is inconsistent"
        )

    # Check that every series contains exactly six yearly rows.
    series_lengths = (
        panel
        .groupby("series_id")
        .size()
    )

    if not (
        series_lengths == 6
    ).all():
        raise ValueError(
            "Every series must contain exactly "
            "six yearly rows"
        )

    # Recalculate the number of observed years in each series.
    calculated_observed_years = (
        panel
        .groupby("series_id")["cell_observed"]
        .transform("sum")
        .astype(int)
    )

    if not (
        calculated_observed_years
        == panel["observed_years_in_series"]
    ).all():
        raise ValueError(
            "The series coverage count "
            "is inconsistent"
        )

    # Recalculate and validate the series coverage rate.
    calculated_coverage_rate = (
        calculated_observed_years / 6
    )

    if not np.allclose(
        calculated_coverage_rate,
        panel["series_coverage_rate"],
        atol=1e-10,
    ):
        raise ValueError(
            "The series coverage rate "
            "is inconsistent"
        )

    # Calculate summary values before using them in formatted strings.
    complete_series_count = panel.loc[
        panel["series_complete_history"],
        "series_id",
    ].nunique()

    total_series_count = (
        panel["series_id"].nunique()
    )

    print(
        "All Q5 panel validation checks passed."
    )

    print(
        f"Rows: {len(panel):,}"
    )

    print(
        f"Boroughs: "
        f"{panel['LA_2023'].nunique()}"
    )

    print(
        f"Observed cells: "
        f"{panel['cell_observed'].sum():,}"
    )

    print(
        f"Unobserved cells: "
        f"{(~panel['cell_observed']).sum():,}"
    )

    print(
        f"Total series: "
        f"{total_series_count:,}"
    )

    print(
        f"Complete six-year series: "
        f"{complete_series_count:,}"
    )
    
validate_q5_panel(
    q5_complete_panel
)

All Q5 panel validation checks passed.
Rows: 4,608
Boroughs: 32
Observed cells: 4,127
Unobserved cells: 481
Total series: 768
Complete six-year series: 570


In [17]:
observed_q5 = q5_complete_panel.loc[
    q5_complete_panel["cell_observed"]
].copy()

# Create temporary diagnostic indicators.
observed_q5["effective_n_below_5"] = (
    observed_q5[
        "effective_n_parent_cell"
    ] < 5
)

observed_q5["effective_n_below_10"] = (
    observed_q5[
        "effective_n_parent_cell"
    ] < 10
)


def summarise_cell_quality(
    data,
    group_columns,
):
    summary = (
        data
        .groupby(
            group_columns,
            observed=True,
            dropna=False,
        )
        .agg(
            observed_cells=(
                "series_id",
                "size",
            ),
            median_raw_n=(
                "n_parent_cell",
                "median",
            ),
            median_effective_n=(
                "effective_n_parent_cell",
                "median",
            ),
            small_cell_rate=(
                "small_cell",
                "mean",
            ),
            few_frequent_cases_rate=(
                "few_frequent_cases",
                "mean",
            ),
            zero_frequent_cases_rate=(
                "zero_frequent_cases",
                "mean",
            ),
            effective_n_below_5_rate=(
                "effective_n_below_5",
                "mean",
            ),
            effective_n_below_10_rate=(
                "effective_n_below_10",
                "mean",
            ),
            mean_cell_frequent_rate=(
                "frequent_volunteer_rate",
                "mean",
            ),
        )
        .reset_index()
    )

    percentage_columns = [
        "small_cell_rate",
        "few_frequent_cases_rate",
        "zero_frequent_cases_rate",
        "effective_n_below_5_rate",
        "effective_n_below_10_rate",
        "mean_cell_frequent_rate",
    ]

    summary[
        percentage_columns
    ] = (
        summary[
            percentage_columns
        ]
        * 100
    )

    return summary


overall_quality_summary = pd.DataFrame(
    {
        "observed_cells": [
            len(observed_q5)
        ],
        "median_raw_n": [
            observed_q5[
                "n_parent_cell"
            ].median()
        ],
        "median_effective_n": [
            observed_q5[
                "effective_n_parent_cell"
            ].median()
        ],
        "small_cell_rate": [
            observed_q5[
                "small_cell"
            ].mean() * 100
        ],
        "few_frequent_cases_rate": [
            observed_q5[
                "few_frequent_cases"
            ].mean() * 100
        ],
        "zero_frequent_cases_rate": [
            observed_q5[
                "zero_frequent_cases"
            ].mean() * 100
        ],
        "effective_n_below_5_rate": [
            observed_q5[
                "effective_n_below_5"
            ].mean() * 100
        ],
        "effective_n_below_10_rate": [
            observed_q5[
                "effective_n_below_10"
            ].mean() * 100
        ],
    }
)

activity_quality_summary = (
    summarise_cell_quality(
        data=observed_q5,
        group_columns=[
            "activity_level_3cat",
            "activity_level_3cat_label",
        ],
    )
    .sort_values(
        "activity_level_3cat"
    )
)

age_quality_summary = (
    summarise_cell_quality(
        data=observed_q5,
        group_columns=[
            "age_group_code",
            "age_group",
        ],
    )
    .sort_values(
        "age_group_code"
    )
)

year_quality_summary = (
    summarise_cell_quality(
        data=observed_q5,
        group_columns=["year"],
    )
    .sort_values("year")
)

borough_quality_summary = (
    summarise_cell_quality(
        data=observed_q5,
        group_columns=["LA_2023"],
    )
    .sort_values(
        "median_effective_n"
    )
)

series_quality = (
    q5_complete_panel[
        [
            "series_id",
            "observed_years_in_series",
            "series_coverage_rate",
            "series_has_history",
            "series_complete_history",
        ]
    ]
    .drop_duplicates(
        subset="series_id"
    )
)

series_coverage_summary = (
    series_quality
    .groupby(
        "observed_years_in_series",
        observed=True,
    )
    .size()
    .rename("number_of_series")
    .reset_index()
    .sort_values(
        "observed_years_in_series"
    )
)

series_coverage_summary[
    "percentage_of_series"
] = (
    series_coverage_summary[
        "number_of_series"
    ]
    / len(series_quality)
    * 100
)

print("Overall cell quality:")
display(overall_quality_summary)

print("\nCell quality by activity level:")
display(activity_quality_summary)

print("\nCell quality by age group:")
display(age_quality_summary)

print("\nCell quality by year:")
display(year_quality_summary)

print("\nCell quality by borough:")
display(borough_quality_summary)

print("\nHistorical coverage by series:")
display(series_coverage_summary)

Overall cell quality:


,observed_cells,median_raw_n,median_effective_n,small_cell_rate,few_frequent_cases_rate,zero_frequent_cases_rate,effective_n_below_5_rate,effective_n_below_10_rate
0,4127,8.0,5.550681,85.31621,88.90235,49.018658,46.23213,69.614732



Cell quality by activity level:


,activity_level_3cat,activity_level_3cat_label,observed_cells,median_raw_n,median_effective_n,small_cell_rate,few_frequent_cases_rate,zero_frequent_cases_rate,effective_n_below_5_rate,effective_n_below_10_rate,mean_cell_frequent_rate
0,0,inactive,1435,7.0,5.051519,98.536585,99.512195,63.135889,49.477352,82.439024,6.003029
1,1,fairly_active,1273,4.0,2.979579,100.0,99.921445,69.206599,75.569521,98.036135,7.718886
2,2,active,1419,24.0,15.734205,58.773784,68.287526,16.631431,16.631431,31.148696,13.370518



Cell quality by age group:


...

,year,observed_cells,median_raw_n,median_effective_n,small_cell_rate,few_frequent_cases_rate,zero_frequent_cases_rate,effective_n_below_5_rate,effective_n_below_10_rate,mean_cell_frequent_rate
0,3,647,4.0,3.116925,96.599691,96.90881,62.44204,66.924266,84.080371,8.129847
1,4,652,5.0,3.557958,95.858896,96.01227,60.582822,64.417178,81.441718,8.472093
2,5,695,9.0,6.291137,80.431655,84.604317,38.273381,36.834532,65.467626,10.614392
3,6,704,10.0,7.046294,80.681818,89.488636,48.4375,38.920455,63.494318,7.399306
4,7,713,10.0,6.986699,80.785414,85.69425,44.460028,36.465638,64.656381,9.232516
5,8,716,10.0,6.970184,79.329609,81.98324,41.899441,37.011173,60.754190,10.419749



Cell quality by borough:


,LA_2023,observed_cells,median_raw_n,median_effective_n,small_cell_rate,few_frequent_cases_rate,zero_frequent_cases_rate,effective_n_below_5_rate,effective_n_below_10_rate,mean_cell_frequent_rate
11,112,130,7.0,4.615105,84.615385,88.461538,58.461538,53.076923,73.076923,6.921392
25,201,126,7.0,4.709268,82.539683,83.333333,45.238095,51.587302,73.015873,12.946927
30,280,119,7.0,4.751622,82.352941,83.193277,50.420168,52.941176,72.268908,10.571167
17,135,130,7.0,4.919789,83.076923,90.0,56.923077,51.538462,72.307692,5.615611
5,44,133,7.0,4.979966,86.466165,91.729323,48.87218,50.375940,71.428571,9.503116
18,136,133,8.0,5.095088,86.466165,90.977444,49.62406,48.872180,69.172932,9.133282
20,142,121,7.0,5.101184,81.818182,90.082645,48.760331,47.107438,71.074380,9.031548
4,35,131,7.0,5.109502,85.496183,89.312977,43.51145,48.091603,74.045802,12.971219
1,9,133,7.0,5.154479,84.962406,87.969925,49.62406,48.120301,72.180451,8.115776
10,109,125,7.0,5.325217,84.8,91.2,58.4,47.200000,71.200000,6.043146



Historical coverage by series:


,observed_years_in_series,number_of_series,percentage_of_series
0,0,10,1.302083
1,1,16,2.083333
2,2,30,3.906250
3,3,23,2.994792
4,4,33,4.296875
5,5,86,11.197917
6,6,570,74.218750


In [18]:
respondent_output_path = (
    OUTPUT_DIR
    / "q5_respondent_master_with_variables.csv"
)

observed_panel_output_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_age_activitylevel_panel_observed.csv"
)

complete_panel_output_path = (
    OUTPUT_DIR
    / "q5_versionA_borough_age_activitylevel_panel_complete.csv"
)

# Create the final observed panel directly from the complete panel.
q5_observed_panel_for_export = (
    q5_complete_panel.loc[
        q5_complete_panel["cell_observed"]
    ]
    .copy()
    .reset_index(drop=True)
)

# Observed rows contain valid non-missing reliability flags.
reliability_flag_columns = [
    "small_cell",
    "few_frequent_cases",
    "zero_frequent_cases",
]

for column in reliability_flag_columns:
    q5_observed_panel_for_export[column] = (
        q5_observed_panel_for_export[column]
        .astype(bool)
    )

q5_master.to_csv(
    respondent_output_path,
    index=False,
)

q5_observed_panel_for_export.to_csv(
    observed_panel_output_path,
    index=False,
)

q5_complete_panel.to_csv(
    complete_panel_output_path,
    index=False,
)

print("Saved respondent-level dataset:")
print(respondent_output_path)
print(f"Rows: {len(q5_master):,}")

print("\nSaved observed panel:")
print(observed_panel_output_path)
print(
    f"Rows: "
    f"{len(q5_observed_panel_for_export):,}"
)

print("\nSaved complete panel:")
print(complete_panel_output_path)
print(f"Rows: {len(q5_complete_panel):,}")

Saved respondent-level dataset:
/Users/zsh/Downloads/PROJECT/code/q5_respondent_master_with_variables.csv
Rows: 96,629

Saved observed panel:
/Users/zsh/Downloads/PROJECT/code/q5_versionA_borough_age_activitylevel_panel_observed.csv
Rows: 4,127

Saved complete panel:
/Users/zsh/Downloads/PROJECT/code/q5_versionA_borough_age_activitylevel_panel_complete.csv
Rows: 4,608
